In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from genpm.modelling.configs import ValidateConfig
from genpm.modelling.model_utils.cvae_utils import (
    generate_timespan,
    load_artifacts,
    load_training_data,
)
from genpm.modelling.validation import (
    compute_kpi_stats,
    enrich_with_window_cols,
    load_real_long,
    plot_autocorr,
    plot_kde,
    plot_timeseries_overlay,
    wide_timespan_to_long,
)

In [ ]:
USER = os.environ.get("USER", "")
SHARED_DIR_PATH = Path(f"/home/{USER}/app/apps/apps/generator/data/shared_dir")
TRAINING_DATA_PATH = SHARED_DIR_PATH / "preprocessed_dataset" / "final_scaled_only_minmax"
RUN_DIR_PATH = SHARED_DIR_PATH / "artifacts" / "run_4"
WEIGHTS_PATH = RUN_DIR_PATH / "models_weights" / "cvae_lstm_v5_0.weights.h5"
REAL_PM_PARQUET = TRAINING_DATA_PATH / "pm_df_long_indexed_winds"

In [ ]:
cfg = ValidateConfig(
    run_dir_path=str(RUN_DIR_PATH),
    weights_path=str(WEIGHTS_PATH),
    real_data_path=str(REAL_PM_PARQUET),
    cell_id="bts_24/cell_5",
    date_start="2023-12-12",
    date_end="2024-03-12",
    kpi_list=["NR_5096", "NR_5244", "NR_5076", "NR_630", "NR_5193", "NR_5324"],
    seed=42,
)

In [ ]:
model, cell_encoder = load_artifacts(
    run_id_path=Path(cfg.run_dir_path),
    weights_path=Path(cfg.weights_path),
)
training_data = load_training_data(Path(cfg.run_dir_path))

In [ ]:
df_syn = generate_timespan(
    model=model,
    **training_data,
    params_df=None,
    cell_id=cfg.cell_id,
    date_start=cfg.date_start,
    date_end=cfg.date_end,
    seed=cfg.seed,
)
df_syn["distname"] = df_syn["cell_id"]
long_syn = wide_timespan_to_long(df_syn)
print(f"Synthetic rows: {len(long_syn):,}")

In [ ]:
long_real = load_real_long(
    parquet_path=cfg.real_data_path,
    distname=cfg.cell_id,
    date_start=cfg.date_start,
    date_end=cfg.date_end,
)

syn_enriched = enrich_with_window_cols(long_syn)
real_enriched = enrich_with_window_cols(long_real)
n_real = real_enriched.groupby(["distname", "window_anchor"]).ngroups
n_syn = syn_enriched.groupby(["distname", "window_anchor"]).ngroups
print(f"Real:      {len(long_real):,} rows  ({n_real} windows)")
print(f"Synthetic: {len(long_syn):,} rows  ({n_syn} windows)")

In [ ]:
display(compute_kpi_stats(long_real, long_syn, cfg.kpi_list))

In [ ]:
plot_timeseries_overlay(long_real, long_syn, cfg.kpi_list, cfg.cell_id, cfg.date_start, cfg.date_end)

In [ ]:
plot_kde(long_real, long_syn, cfg.kpi_list)

In [ ]:
plot_autocorr(long_real, long_syn, cfg.kpi_list)